# Medicine Label Helper: train your own small AI

**Teaching demo only. Not medical advice.**

Press the play button on each gray box (a *cell*) **in order, top to bottom**. Wait for each one to finish (a green check mark appears to its left) before pressing the next. You do not need to change anything.

## Step 1. Check for a GPU
A *GPU* is a fast chip that makes training quicker. You should see **GPU found: Tesla T4** (or another name). If you see *No GPU*, go to the menu **Runtime > Change runtime type**, pick **T4 GPU**, click **Save**, then press play here again.

In [ ]:
import torch
print('GPU found:', torch.cuda.get_device_name(0)) if torch.cuda.is_available() else print('No GPU. See the note above.')

## Step 2. Download the project and install tools (about 1-2 minutes)
You will see many lines of text scroll by. That is normal. It is done when the last line says **Ready.**

In [ ]:
!git clone -q https://github.com/serewicz/med-label-helper.git 2>/dev/null || echo 'Already downloaded'
%cd med-label-helper
!pip install -q -r requirements.txt
print('Ready.')

## Step 3. Look at a few training examples
These are made-up medicine labels with plain-language explanations. The AI learns from 300 of these.

In [ ]:
import json
rows = [json.loads(l) for l in open('data/train.jsonl')]
print(len(rows), 'examples')
for r in rows[:3]:
    print('\nQUESTION:', r['messages'][1]['content'])
    print('ANSWER:  ', r['messages'][2]['content'])

## Step 4. Ask the original model (before training)
This shows how the model answers *before* we teach it. It downloads the model the first time (about 1 GB, 1-2 minutes).

In [ ]:
import sys; sys.path.insert(0, 'scripts')
from helper import load, ask
QUESTION = '''What does this label mean?

Sunrise Pharmacy
Brenolol 25 mg tablets
TAKE 1 TABLET BY MOUTH TWICE DAILY
TAKE WITH FOOD
Refills: 2'''
tok, model = load()
print(ask(tok, model, QUESTION))
del model

## Step 5. Train (about 5-10 minutes on a T4 GPU)
You will see lines with **loss** numbers. *Loss* is a score of how wrong the model is; it should go down. It is done when you see **Done. Saved the LoRA adapter**.

In [ ]:
!python scripts/train.py

## Step 6. Ask the trained model the same question
Compare this answer with Step 4.

In [ ]:
tok, model = load(adapter='med-label-lora')
print(ask(tok, model, QUESTION))

## Step 7. Score both models on 20 new questions (about 3-5 minutes)
At the end you will see a small table. See the README for what each column means.

In [ ]:
!python scripts/evaluate.py --adapter med-label-lora

## Step 8. Try it in a chat page
After about a minute you will see a link ending in **gradio.live**. Click it to open a page where you can paste any label. To stop, press the square stop button to the left of this cell.

In [ ]:
!python app.py --share